# Array-Only Methylation deepTools Prototype


## 1. Environment and imports

In [55]:
%load_ext autoreload
%autoreload 2

import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "repo_paths.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate repo root from the current working directory.")
    PROJECT_ROOT = PROJECT_ROOT.parent

CHROMATIN_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "03_chromatin_analysis"
REGION_CALLING_UTILS_DIR = PROJECT_ROOT / "analysis" / "01_region_calling_analysis" / "utils"
matplotlib_cache_dir = Path(tempfile.gettempdir()) / f"matplotlib-{os.getuid()}"
matplotlib_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(matplotlib_cache_dir)

for import_path in [PROJECT_ROOT, CHROMATIN_ANALYSIS_DIR, REGION_CALLING_UTILS_DIR]:
    import_str = str(import_path)
    if import_str not in sys.path:
        sys.path.insert(0, import_str)

from repo_paths import REGION_CALLING_RESULTS_DIR
from methyl_tool_comparator import SharedPrepManager
from chromatin_analysis_utils import (
    clean_interval_df,
    filter_regions_for_deeptools,
    get_deeptools_processor_count,
    get_eligible_chrom_sizes,
    run_command,
    sample_to_sample_id,
    write_bed,
)
from figures.utils.figures_utils import (
    REGION_CALLING_TOOL_LABELS,
    TOOL_REGISTRY as FIGURE_TOOL_REGISTRY,
    _canonical_tool_name,
    _load_chrom_sizes,
    _write_bigwig,
    load_tool_regions,
    region_type_for_tool,
)

CANONICAL_CHROMOSOMES = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
FIGURE_TOOL_REGISTRY_BY_NAME = {
    tool_config["tool"]: tool_config for tool_config in FIGURE_TOOL_REGISTRY
}
TOOL_PLATFORM_BY_NAME = {
    tool_name: tool_config["platform"] for tool_name, tool_config in FIGURE_TOOL_REGISTRY_BY_NAME.items()
}
ARRAY_MODE_TOOLS = ["methylseg_hm450k", "dnmtools_array"]


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. User knobs


In [56]:
CONFIGS_PATH = PROJECT_ROOT / "analysis" / "01_region_calling_analysis" / "slurm_code" / "configs.txt"
OUT_DIR = PROJECT_ROOT / "analysis" / "01_region_calling_analysis" / "out" / "methylation_deeptools_test_array_only"

SELECTED_SAMPLES = ["SRR26107673"]
SELECTED_TOOLS = ["methylseg_hm450k", "dnmtools_array"]

INCLUDE_HEATMAPS = False
COMPUTE_MATRIX_BIN_SIZE = 10_000
REGION_BODY_LENGTH = 500_000
ARRAY_FLANK_LENGTH = 250_000

FORCE_REBUILD_BIGWIGS = True
FORCE_RERUN_DEEPTOOLS = True

PREVIEW_SAMPLE = ["SRR26107673"]

BIGWIG_DIR = OUT_DIR / "bigwigs"
PREPARED_REGION_DIR = OUT_DIR / "prepared_regions"
DEEPTOOLS_DIR = OUT_DIR / "deeptools"
SHARED_PREP_DIR = OUT_DIR / "shared_prep"
TABLES_DIR = OUT_DIR / "tables"
LOGS_DIR = OUT_DIR / "logs"
DEEPTOOLS_EXPECTED_ENV_BIN = Path(
    "/uufs/chpc.utah.edu/common/home/clementm-group1/conda/mambaforge/env/jt_wgbs_analysis/bin"
)

for directory in [OUT_DIR, BIGWIG_DIR, PREPARED_REGION_DIR, DEEPTOOLS_DIR, SHARED_PREP_DIR, TABLES_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

display(Markdown(f"Using output root: `{OUT_DIR}`"))


Using output root: `/uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test_array_only`

## 3. Sample/config discovery


In [57]:
def ensure_deeptools_commands() -> dict[str, str]:
    commands = {
        "computeMatrix": shutil.which("computeMatrix"),
        "plotProfile": shutil.which("plotProfile"),
        "plotHeatmap": shutil.which("plotHeatmap"),
    }
    missing = [name for name, resolved in commands.items() if resolved is None]
    if missing:
        expected_msg = ""
        if DEEPTOOLS_EXPECTED_ENV_BIN.exists():
            expected_msg = (
                f" Expected them on PATH from jt_wgbs_analysis, for example under {DEEPTOOLS_EXPECTED_ENV_BIN}."
            )
        raise RuntimeError(
            "Missing deepTools command(s): " + ", ".join(missing) + "."
            + " Start the notebook from the jt_wgbs_analysis environment."
            + expected_msg
        )
    return commands


def validate_selected_tools(selected_tools: list[str]) -> list[str]:
    available_tools = list(ARRAY_MODE_TOOLS)
    deduped_tools = list(dict.fromkeys(selected_tools))
    invalid = [tool for tool in deduped_tools if tool not in available_tools]
    if invalid:
        raise ValueError(
            "Unsupported selected tools: " + ", ".join(invalid)
            + ". Choose from: " + ", ".join(available_tools)
        )
    if not deduped_tools:
        raise ValueError("SELECTED_TOOLS must contain at least one tool.")
    return deduped_tools


def read_config_paths(configs_path: Path) -> list[Path]:
    configs_path = Path(configs_path).expanduser().resolve()
    if not configs_path.exists():
        raise FileNotFoundError(f"Config manifest not found: {configs_path}")
    config_paths = []
    for raw_line in configs_path.read_text().splitlines():
        line = raw_line.strip()
        if not line:
            continue
        config_path = Path(line).expanduser().resolve()
        if not config_path.exists():
            raise FileNotFoundError(f"Config file listed in {configs_path} does not exist: {config_path}")
        config_paths.append(config_path)
    if not config_paths:
        raise ValueError(f"No config paths were found in {configs_path}")
    return config_paths


def load_sample_configs(configs_path: Path, selected_samples=None) -> pd.DataFrame:
    selected_sample_set = None if selected_samples is None else {str(sample) for sample in selected_samples}
    rows = []
    for config_order, config_path in enumerate(read_config_paths(configs_path)):
        with open(config_path) as handle:
            config = yaml.safe_load(handle) or {}
        sample = str(config.get("sample", "")).strip()
        meth_file = str(config.get("meth_file", "")).strip()
        genome = str(config.get("genome", "")).strip()
        if not sample or not meth_file or not genome:
            raise ValueError(
                f"Config is missing one of sample/meth_file/genome: {config_path}"
            )
        if selected_sample_set is not None and sample not in selected_sample_set:
            continue
        rows.append(
            {
                "config_order": config_order,
                "config_path": str(config_path),
                "sample": sample,
                "sample_id": sample_to_sample_id(sample),
                "meth_file": str(Path(meth_file).expanduser().resolve()),
                "genome": genome,
            }
        )
    if not rows:
        raise ValueError("No sample configs matched the current selection.")
    return pd.DataFrame(rows).sort_values("config_order").reset_index(drop=True)


def load_beta_track_from_beta_table(beta_path: Path) -> pd.DataFrame:
    beta_path = Path(beta_path).expanduser().resolve()
    if not beta_path.exists():
        raise FileNotFoundError(f"Missing beta track file: {beta_path}")

    with open(beta_path) as fh:
        header_fields = fh.readline().rstrip("\n").split("\t")

    has_header = header_fields[:4] == ["chrom", "start", "end", "beta"]
    if has_header:
        beta_df = pd.read_csv(beta_path, sep="\t")
    else:
        n_cols = len(header_fields)
        if n_cols < 4:
            raise ValueError(f"Unsupported beta track with fewer than 4 columns: {beta_path}")
        column_names = ["chrom", "start", "end", "beta"] + [f"extra_{idx}" for idx in range(n_cols - 4)]
        beta_df = pd.read_csv(beta_path, sep="\t", header=None, names=column_names)

    beta_df = beta_df.loc[:, ["chrom", "start", "end", "beta"]].copy()
    beta_df["chrom"] = beta_df["chrom"].astype(str)
    beta_df["chrom"] = "chr" + beta_df["chrom"].str.replace("^chr", "", regex=True)
    beta_df["start"] = pd.to_numeric(beta_df["start"], errors="coerce")
    beta_df["end"] = pd.to_numeric(beta_df["end"], errors="coerce")
    beta_df["beta"] = pd.to_numeric(beta_df["beta"], errors="coerce")
    beta_df = beta_df.dropna(subset=["chrom", "start", "end", "beta"]).copy()
    beta_df = beta_df.loc[beta_df["chrom"].isin(CANONICAL_CHROMOSOMES)].copy()
    beta_df = beta_df.loc[beta_df["beta"].between(0.0, 1.0)].copy()
    beta_df["start"] = beta_df["start"].astype(int)
    beta_df["end"] = beta_df["end"].astype(int)
    beta_df = beta_df.loc[beta_df["end"] > beta_df["start"]].reset_index(drop=True)
    return beta_df


def build_shared_prep_outputs(config_row: dict):
    manager = SharedPrepManager(
        sample_id=config_row["sample"],
        meth_file=config_row["meth_file"],
        genome=config_row["genome"],
        out_dir=SHARED_PREP_DIR,
        force_recreate=FORCE_REBUILD_BIGWIGS,
        print_logs=True,
        skip_450k=False,
    )
    return manager.prepare()


def resolve_source_region_path(sample: str, tool: str) -> Path:
    canonical_tool = _canonical_tool_name(tool)
    tool_config = FIGURE_TOOL_REGISTRY_BY_NAME[canonical_tool]
    source_path = REGION_CALLING_RESULTS_DIR
    for part in tool_config["path_parts"]:
        source_path = source_path / part.format(sample=sample)
    return source_path


def sort_manifest_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    sort_cols = [col for col in ["config_order", "tool_order", "sample", "tool"] if col in df.columns]
    if not sort_cols:
        return df.reset_index(drop=True)
    return df.sort_values(sort_cols).reset_index(drop=True)


deeptools_commands = ensure_deeptools_commands()
selected_tools = validate_selected_tools(SELECTED_TOOLS)
sample_configs_df = load_sample_configs(CONFIGS_PATH, SELECTED_SAMPLES)
signal_track = "hm450k"

display(sample_configs_df)
print("Selected tools:", selected_tools)
print("Signal track:", signal_track)
print("Samples:", sample_configs_df["sample"].tolist())


,config_order,config_path,sample,sample_id,meth_file,genome
0,0,/uufs/chpc.utah.edu/common/home/clementm-group...,SRR26107673,SRR26107673,/uufs/chpc.utah.edu/common/home/clementm-group...,hg19


Selected tools: ['methylseg_hm450k', 'dnmtools_array']
Signal track: hm450k
Samples: ['SRR26107673']


## 4. Signal-track export


In [58]:
def export_probe_level_bigwig(config_row: dict, shared_prep_outputs) -> dict:
    sample = config_row["sample"]
    output_path = BIGWIG_DIR / f"{sample}.hm450k.probe_level.methylation.bigwig"
    cache_used = output_path.exists() and not FORCE_REBUILD_BIGWIGS

    if shared_prep_outputs is None or shared_prep_outputs.hm450k_beta is None:
        raise FileNotFoundError(
            f"HM450K shared prep beta track is missing for {sample}."
        )

    signal_source = str(Path(shared_prep_outputs.hm450k_beta).expanduser().resolve())
    if not cache_used:
        beta_df = load_beta_track_from_beta_table(signal_source)
        chrom_sizes = _load_chrom_sizes(config_row["genome"])
        _write_bigwig(beta_df, output_path, chrom_sizes)

    return {
        "config_order": int(config_row["config_order"]),
        "sample": sample,
        "sample_id": config_row["sample_id"],
        "genome": config_row["genome"],
        "signal_track": "hm450k",
        "signal_mode": "probe_level",
        "signal_source": signal_source,
        "compute_matrix_bin_bp": int(COMPUTE_MATRIX_BIN_SIZE),
        "flank_length": int(ARRAY_FLANK_LENGTH),
        "bigwig_path": str(output_path),
        "cache_used": cache_used,
    }


signal_rows = []
sample_failures = []

for config_row in sample_configs_df.to_dict("records"):
    try:
        shared_prep_outputs = build_shared_prep_outputs(config_row)
        signal_rows.append(export_probe_level_bigwig(config_row, shared_prep_outputs))
    except Exception as exc:
        sample_failures.append(
            {
                "sample": config_row["sample"],
                "tool": "<sample-level>",
                "stage": "signal_track_export",
                "error": str(exc),
            }
        )

signal_manifest_df = pd.DataFrame(signal_rows)
if not signal_manifest_df.empty:
    signal_manifest_df = signal_manifest_df.sort_values(["config_order", "sample"]).reset_index(drop=True)

signal_manifest_path = TABLES_DIR / "signal_manifest.tsv"
if not signal_manifest_df.empty:
    signal_manifest_df.to_csv(signal_manifest_path, sep="	", index=False)

display(signal_manifest_df)
print(f"Saved signal manifest to: {signal_manifest_path}")



[2026-08-17 13:26:39] Building shared prep artifacts in /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test_array_only/shared_prep/SRR26107673/shared_prep
[2026-08-17 13:27:02] Command wgbstools view --genome hg19 /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/data/methylation_data/SRR26107673.beta -o /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test_array_only/shared_prep/SRR26107673/shared_prep/.wgbs.tsv.tmp.2378328.1786994799299397150 | Output: /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test_array_only/shared_prep/SRR26107673/logs/job_logs/wgbstools_2247635.stdout | Error: /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analys

,config_order,sample,sample_id,genome,signal_track,signal_mode,signal_source,compute_matrix_bin_bp,flank_length,bigwig_path,cache_used
0,0,SRR26107673,SRR26107673,hg19,hm450k,probe_level,/uufs/chpc.utah.edu/common/home/clementm-group...,10000,250000,/uufs/chpc.utah.edu/common/home/clementm-group...,False


Saved signal manifest to: /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test_array_only/tables/signal_manifest.tsv


## 5. Region preparation


In [59]:
def build_region_manifest_row(config_row: dict, tool: str, signal_row: dict) -> dict:
    source_region_path = resolve_source_region_path(config_row["sample"], tool)
    if not source_region_path.exists():
        raise FileNotFoundError(
            f"Missing region file for {config_row['sample']} {tool}: {source_region_path}"
        )

    bigwig_path = Path(signal_row["bigwig_path"])
    chrom_sizes = get_eligible_chrom_sizes(CANONICAL_CHROMOSOMES, bigwig_path)
    if not chrom_sizes:
        raise RuntimeError(f"No eligible canonical chromosomes found in {bigwig_path}")

    raw_region_df = load_tool_regions(config_row["sample"], tool)
    prepared_region_df = clean_interval_df(raw_region_df, chrom_sizes)
    if prepared_region_df.empty:
        raise AssertionError(
            f"{config_row['sample']} {tool} produced zero retained regions after cleaning."
        )

    prepared_path = PREPARED_REGION_DIR / config_row["sample"] / f"{tool}.bed"
    prepared_path.parent.mkdir(parents=True, exist_ok=True)
    write_bed(prepared_region_df, prepared_path)

    deeptools_region_df = filter_regions_for_deeptools(prepared_region_df, int(COMPUTE_MATRIX_BIN_SIZE))
    deeptools_region_path = (
        DEEPTOOLS_DIR
        / config_row["sample"]
        / tool
        / f"{config_row['sample']}.{tool}.deeptools_regions.bed"
    )
    deeptools_region_path.parent.mkdir(parents=True, exist_ok=True)
    write_bed(deeptools_region_df, deeptools_region_path)

    return {
        "config_order": int(config_row["config_order"]),
        "tool_order": int(selected_tools.index(tool)),
        "sample": config_row["sample"],
        "sample_id": config_row["sample_id"],
        "tool": tool,
        "tool_label": REGION_CALLING_TOOL_LABELS[tool],
        "tool_platform": TOOL_PLATFORM_BY_NAME[_canonical_tool_name(tool)],
        "signal_track": signal_row["signal_track"],
        "signal_mode": signal_row["signal_mode"],
        "signal_source": signal_row["signal_source"],
        "region_type": region_type_for_tool(tool),
        "source_region_path": str(source_region_path),
        "prepared_region_path": str(prepared_path),
        "deeptools_region_path": str(deeptools_region_path),
        "bigwig_path": str(bigwig_path),
        "total_regions": int(len(prepared_region_df)),
        "visualized_regions": int(len(deeptools_region_df)),
        "excluded_short_regions": int(len(prepared_region_df) - len(deeptools_region_df)),
        "min_region_length_bp": int(COMPUTE_MATRIX_BIN_SIZE),
        "compute_matrix_bin_bp": int(COMPUTE_MATRIX_BIN_SIZE),
        "flank_length": int(ARRAY_FLANK_LENGTH),
        "region_body_length": int(REGION_BODY_LENGTH),
    }


signal_lookup = {row["sample"]: row for row in signal_rows}
region_rows = []
failure_rows = list(sample_failures)

for config_row in sample_configs_df.to_dict("records"):
    for tool in selected_tools:
        try:
            signal_row = signal_lookup.get(config_row["sample"])
            if signal_row is None:
                raise FileNotFoundError(
                    f"Missing hm450k signal track for {config_row['sample']} {tool}."
                )
            region_rows.append(build_region_manifest_row(config_row, tool, signal_row))
        except Exception as exc:
            failure_rows.append(
                {
                    "sample": config_row["sample"],
                    "tool": tool,
                    "stage": "region_preparation",
                    "error": str(exc),
                }
            )

region_manifest_df = sort_manifest_df(pd.DataFrame(region_rows))
region_manifest_path = TABLES_DIR / "region_manifest.tsv"
if not region_manifest_df.empty:
    region_manifest_df.to_csv(region_manifest_path, sep="\t", index=False)

display(region_manifest_df)
print(f"Saved region manifest to: {region_manifest_path}")


,config_order,tool_order,sample,sample_id,tool,tool_label,tool_platform,signal_track,signal_mode,signal_source,...,prepared_region_path,deeptools_region_path,bigwig_path,total_regions,visualized_regions,excluded_short_regions,min_region_length_bp,compute_matrix_bin_bp,flank_length,region_body_length
0,0,0,SRR26107673,SRR26107673,methylseg_hm450k,MethylSeg HM450K,hm450k,hm450k,probe_level,/uufs/chpc.utah.edu/common/home/clementm-group...,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,905,904,1,10000,10000,250000,500000
1,0,1,SRR26107673,SRR26107673,dnmtools_array,DNMTools Array,hm450k,hm450k,probe_level,/uufs/chpc.utah.edu/common/home/clementm-group...,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,53,53,0,10000,10000,250000,500000


Saved region manifest to: /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test_array_only/tables/region_manifest.tsv


## 6. deepTools execution


In [60]:
def maybe_run_command(command, expected_outputs, force=False):
    expected_paths = [Path(path) for path in expected_outputs]
    if not force and expected_paths and all(path.exists() for path in expected_paths):
        return True
    run_command(command, expected_outputs=expected_paths)
    return False


def run_deeptools_for_region(region_row: dict) -> dict:
    if int(region_row["visualized_regions"]) <= 0:
        raise RuntimeError(
            f"{region_row['sample']} {region_row['tool']} had zero regions at least {region_row['compute_matrix_bin_bp']} bp after filtering."
        )

    sample = str(region_row["sample"])
    sample_id = str(region_row["sample_id"])
    tool = str(region_row["tool"])
    tool_label = str(region_row["tool_label"])
    sample_output_dir = DEEPTOOLS_DIR / sample / tool
    sample_output_dir.mkdir(parents=True, exist_ok=True)

    matrix_path = sample_output_dir / f"{sample}.{tool}.methylation.matrix.gz"
    matrix_values_path = sample_output_dir / f"{sample}.{tool}.methylation.matrix.tsv"
    sorted_regions_path = sample_output_dir / f"{sample}.{tool}.sorted_regions.bed"
    profile_path = sample_output_dir / f"{sample}.{tool}.profile.png"
    heatmap_path = sample_output_dir / f"{sample}.{tool}.heatmap.png"

    profile_title = f"{sample}: methylation across {tool_label} called regions"
    heatmap_title = f"{sample}: methylation heatmap across {tool_label} called regions"

    compute_matrix_command = [
        deeptools_commands["computeMatrix"],
        "scale-regions",
        "-p",
        str(get_deeptools_processor_count()),
        "-S",
        str(region_row["bigwig_path"]),
        "-R",
        str(region_row["deeptools_region_path"]),
        "-b",
        str(region_row["flank_length"]),
        "-a",
        str(region_row["flank_length"]),
        "--binSize",
        str(region_row["compute_matrix_bin_bp"]),
        "--regionBodyLength",
        str(REGION_BODY_LENGTH),
        "--sortRegions",
        "keep",
        "-o",
        str(matrix_path),
        "--outFileSortedRegions",
        str(sorted_regions_path),
        "--outFileNameMatrix",
        str(matrix_values_path),
    ]
    compute_matrix_cache_used = maybe_run_command(
        compute_matrix_command,
        [matrix_path, matrix_values_path, sorted_regions_path],
        force=FORCE_RERUN_DEEPTOOLS,
    )

    profile_command = [
        deeptools_commands["plotProfile"],
        "--numPlotsPerRow",
        "1",
        "-m",
        str(matrix_path),
        "--perGroup",
        "--averageType",
        "median",
        "--samplesLabel",
        sample_id,
        "--regionsLabel",
        tool_label,
        "--startLabel",
        "Region start",
        "--endLabel",
        "Region end",
        "--plotTitle",
        profile_title,
        "--plotWidth",
        "11",
        "--plotHeight",
        "6",
        "--dpi",
        "200",
        "-out",
        str(profile_path),
    ]
    profile_cache_used = maybe_run_command(
        profile_command,
        [profile_path],
        force=FORCE_RERUN_DEEPTOOLS,
    )

    heatmap_cache_used = None
    saved_heatmap_path = ""
    if INCLUDE_HEATMAPS:
        heatmap_command = [
            deeptools_commands["plotHeatmap"],
            "-m",
            str(matrix_path),
            "--samplesLabel",
            sample_id,
            "--regionsLabel",
            tool_label,
            "--startLabel",
            "Region start",
            "--endLabel",
            "Region end",
            "--plotTitle",
            heatmap_title,
            "--sortRegions",
            "keep",
            "--heatmapWidth",
            "12",
            "--heatmapHeight",
            "10",
            "--whatToShow",
            "heatmap and colorbar",
            "-out",
            str(heatmap_path),
        ]
        heatmap_cache_used = maybe_run_command(
            heatmap_command,
            [heatmap_path],
            force=FORCE_RERUN_DEEPTOOLS,
        )
        saved_heatmap_path = str(heatmap_path)

    return {
        **region_row,
        "matrix_path": str(matrix_path),
        "matrix_values_path": str(matrix_values_path),
        "sorted_regions_path": str(sorted_regions_path),
        "profile_path": str(profile_path),
        "heatmap_path": saved_heatmap_path,
        "profile_title": profile_title,
        "heatmap_title": heatmap_title,
        "compute_matrix_cache_used": compute_matrix_cache_used,
        "profile_cache_used": profile_cache_used,
        "heatmap_cache_used": heatmap_cache_used,
    }


deeptools_rows = []
for region_row in region_manifest_df.to_dict("records"):
    try:
        deeptools_rows.append(run_deeptools_for_region(region_row))
    except Exception as exc:
        failure_rows.append(
            {
                "sample": region_row["sample"],
                "tool": region_row["tool"],
                "stage": "deeptools_execution",
                "error": str(exc),
            }
        )

deeptools_outputs_df = sort_manifest_df(pd.DataFrame(deeptools_rows))
deeptools_outputs_path = TABLES_DIR / "deeptools_outputs.tsv"
if not deeptools_outputs_df.empty:
    deeptools_outputs_df.to_csv(deeptools_outputs_path, sep="	", index=False)

if deeptools_outputs_df.empty:
    display(deeptools_outputs_df)
else:
    display(
        deeptools_outputs_df[
            [
                "sample",
                "tool",
                "tool_label",
                "signal_track",
                "signal_mode",
                "compute_matrix_bin_bp",
                "flank_length",
                "matrix_path",
                "profile_path",
                "heatmap_path",
                "visualized_regions",
            ]
        ]
    )
print(f"Saved deepTools outputs manifest to: {deeptools_outputs_path}")



$ /uufs/chpc.utah.edu/common/home/clementm-group1/conda/mambaforge/env/jt_wgbs_analysis/bin/computeMatrix scale-regions -p 1 -S /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test_array_only/bigwigs/SRR26107673.hm450k.probe_level.methylation.bigwig -R /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test_array_only/deeptools/SRR26107673/methylseg_hm450k/SRR26107673.methylseg_hm450k.deeptools_regions.bed -b 250000 -a 250000 --binSize 10000 --regionBodyLength 500000 --sortRegions keep -o /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test_array_only/deeptools/SRR26107673/methylseg_hm450k/SRR26107673.methylseg_hm450k.methylation.matrix.gz --outFileSortedRegions /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260

,sample,tool,tool_label,signal_track,signal_mode,compute_matrix_bin_bp,flank_length,matrix_path,profile_path,heatmap_path,visualized_regions
0,SRR26107673,methylseg_hm450k,MethylSeg HM450K,hm450k,probe_level,10000,250000,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,,904
1,SRR26107673,dnmtools_array,DNMTools Array,hm450k,probe_level,10000,250000,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,,53


Saved deepTools outputs manifest to: /uufs/chpc.utah.edu/common/home/clementm-group1/projects/20260624_methylseg/analysis/01_region_calling_analysis/out/methylation_deeptools_test_array_only/tables/deeptools_outputs.tsv


## 7. Manifest summaries and inline display


In [61]:
failures_df = pd.DataFrame(failure_rows)

expected_combinations = len(sample_configs_df) * len(selected_tools)
successful_combinations = len(deeptools_outputs_df)
print(f"Successful sample/tool combinations: {successful_combinations} / {expected_combinations}")

display(Markdown("### Signal manifest"))
display(signal_manifest_df)
display(Markdown("### Region manifest"))
display(region_manifest_df)
display(Markdown("### deepTools outputs manifest"))
display(deeptools_outputs_df)

if failures_df.empty:
    display(Markdown("### Failures\nNo sample/tool failures were recorded."))
else:
    display(Markdown("### Failures"))
    display(failures_df)

preview_candidates = deeptools_outputs_df["sample"].drop_duplicates().tolist()
preview_sample = PREVIEW_SAMPLE if PREVIEW_SAMPLE is not None else (preview_candidates[0] if preview_candidates else None)

if preview_sample is None:
    display(Markdown("### Preview\nNo successful deepTools outputs are available to preview yet."))
else:
    preview_df = deeptools_outputs_df.loc[
        deeptools_outputs_df["sample"].astype(str) == str(preview_sample)
    ].copy()
    preview_df = preview_df.sort_values(["tool_order", "tool"]).reset_index(drop=True)
    display(Markdown(f"### Preview sample: `{preview_sample}`"))
    for row in preview_df.itertuples(index=False):
        display(Markdown(f"#### {row.tool_label} ({row.region_type.upper()}, signal={row.signal_track}, mode={row.signal_mode})"))
        display(Markdown(f"Matrix: `{row.matrix_path}`"))
        if Path(row.profile_path).exists():
            display(Image(filename=row.profile_path))
        else:
            display(Markdown(f"Missing profile image: `{row.profile_path}`"))
        if INCLUDE_HEATMAPS and str(row.heatmap_path).strip():
            if Path(row.heatmap_path).exists():
                display(Image(filename=row.heatmap_path))
            else:
                display(Markdown(f"Missing heatmap image: `{row.heatmap_path}`"))


Successful sample/tool combinations: 2 / 2


### Signal manifest

,config_order,sample,sample_id,genome,signal_track,signal_mode,signal_source,compute_matrix_bin_bp,flank_length,bigwig_path,cache_used
0,0,SRR26107673,SRR26107673,hg19,hm450k,probe_level,/uufs/chpc.utah.edu/common/home/clementm-group...,10000,250000,/uufs/chpc.utah.edu/common/home/clementm-group...,False


### Region manifest

,config_order,tool_order,sample,sample_id,tool,tool_label,tool_platform,signal_track,signal_mode,signal_source,...,prepared_region_path,deeptools_region_path,bigwig_path,total_regions,visualized_regions,excluded_short_regions,min_region_length_bp,compute_matrix_bin_bp,flank_length,region_body_length
0,0,0,SRR26107673,SRR26107673,methylseg_hm450k,MethylSeg HM450K,hm450k,hm450k,probe_level,/uufs/chpc.utah.edu/common/home/clementm-group...,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,905,904,1,10000,10000,250000,500000
1,0,1,SRR26107673,SRR26107673,dnmtools_array,DNMTools Array,hm450k,hm450k,probe_level,/uufs/chpc.utah.edu/common/home/clementm-group...,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,53,53,0,10000,10000,250000,500000


### deepTools outputs manifest

,config_order,tool_order,sample,sample_id,tool,tool_label,tool_platform,signal_track,signal_mode,signal_source,...,matrix_path,matrix_values_path,sorted_regions_path,profile_path,heatmap_path,profile_title,heatmap_title,compute_matrix_cache_used,profile_cache_used,heatmap_cache_used
0,0,0,SRR26107673,SRR26107673,methylseg_hm450k,MethylSeg HM450K,hm450k,hm450k,probe_level,/uufs/chpc.utah.edu/common/home/clementm-group...,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,,SRR26107673: methylation across MethylSeg HM45...,SRR26107673: methylation heatmap across Methyl...,False,False,None
1,0,1,SRR26107673,SRR26107673,dnmtools_array,DNMTools Array,hm450k,hm450k,probe_level,/uufs/chpc.utah.edu/common/home/clementm-group...,...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,/uufs/chpc.utah.edu/common/home/clementm-group...,,SRR26107673: methylation across DNMTools Array...,SRR26107673: methylation heatmap across DNMToo...,False,False,None


### Failures
No sample/tool failures were recorded.

### Preview sample: `['SRR26107673']`